Modeling: Fluxes
================

A measurable quantity of a point source is its flux—the total amount of light received from each multiple image
of the point source (e.g., the quasar images).

In practice, fluxes are often measured but not used directly when analyzing lensed point sources such as quasars or
supernovae. This is because fluxes can be significantly affected by microlensing, which many lens models do not
accurately capture. However, in this simulation, microlensing is not included, so the fluxes can be simulated and
fitted reliably.

Nevertheless, this script describes how to perform point source lens modeling using the fluxes of the point source
dataset as additional information on top of the positions of the point source, in case you are studying microlensing
or confident the fluxes are not affected by it.

__Contents__

- **Model:** Compose the lens model fitted to the data.
- **Dataset:** Load and plot the strong lens dataset.
- **Point Solver:** We set up the `PointSolver`, which is used to compute the multiple images of the point source in.
- **Search:** Configure the non-linear search used to fit the model.
- **Analysis:** Create the Analysis object that defines how the model is fitted to the data.
- **Run Times:** Profiling the expected run time of the model-fit.
- **Result:** Overview of the results of the model-fit.

__Model__

This script fits a `PointDataset` data of a 'galaxy-scale' strong lens with a model where:

 - The lens galaxy's total mass distribution is an `Isothermal`.
 - The source `Galaxy` is a point source with flux, a `PointFlux`.

No external shear is included in the model, where it is for the `imaging` and `interferometer` examples. For a
quadruply imaged point source (8 data points) there is insufficient information to constrain an `Isothermal` plus
an external shear (9 parameters).

Where a shear is fitted, it is never attached to the lens galaxy: it is an `ExternalShear` held in an
`al.MassField`, which lives in the model's own `fields` slot beside `galaxies=` (and in the tracer's
`fields=` argument). The shear describes the tidal field of everything outside the system, so it is a property of
the system rather than of a galaxy. See `imaging/modeling.py` for the full description.

__Start Here Notebook__

If any code in this script is unclear, refer to the `point_source/start_here.ipynb` notebook.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/autolens_workspace/blob/main/start_here.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Dataset__

Load the strong lens point-source dataset `simple`, which is the dataset we will use to perform point source 
lens modeling.

In [ ]:
dataset_name = "simple"
dataset_path = Path("dataset") / "point_source" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/point_source/simulator.py"],
        check=True,
    )

We now load the point source dataset we will fit using point source modeling. 

We load this data as a `PointDataset`, which contains the positions and fluxes of every point source. 

In [ ]:
dataset = al.from_json(
    file_path=dataset_path / "point_dataset_with_fluxes.json",
)

We can print this dictionary to see the dataset's `name`, `positions` and `fluxes` and noise-map values.

In [ ]:
print("Point Dataset Info:")
print(dataset.info)

We can also plot the positions and fluxes of the `PointDataset`.

In [ ]:
aplt.subplot_point_dataset(dataset=dataset)

We next load an image of the dataset, because as described in the `point_source/start_here.ipynb` notebook, it is
useful for visualizing the point source dataset.

It is also passed to the analysis below, where it is used only for visualization and not in the likelihood:
`image_with_points.png` shows the data with the observed positions (before the fit) and `fit_image_with_points.png`
adds the model positions and critical curves.

In [ ]:
data = al.Array2D.from_fits(file_path=dataset_path / "data.fits", pixel_scales=0.05)


aplt.plot_array(array=data, title="")

__Point Solver__

We set up the `PointSolver`, which is used to compute the multiple images of the point source in the image-plane.

There are no special settings or inputs for the fitting of fluxes, therefore the `PointSolver` is set up in the same way
as in the `point_source/start_here.ipynb` notebook.

In [ ]:
grid = al.Grid2D.uniform(
    shape_native=(100, 100),
    pixel_scales=0.2,  # <- The pixel-scale describes the conversion from pixel units to arc-seconds.
)

solver = al.PointSolver.for_grid(
    grid=grid, pixel_scale_precision=0.001, magnification_threshold=0.1
)

__Model__

We compose a lens model where:

 - The lens galaxy's total mass distribution is an `Isothermal` [5 parameters].
 - The source galaxy's light is a point `PointFlux` [3 parameters].

The number of free parameters and therefore the dimensionality of non-linear parameter space is N=8.

Name pairing is used as before to pair the `PointDataset` to the `Point` in the model, which is discussed below.

To fit fluxes, our model point source also needs a flux parameter, which is done by using the `PointFlux`
component instead of the `Point` component. This has a free parameter `flux`, which is the flux of the point source
in the source-plane. 

In [ ]:
# Lens:

mass = af.Model(al.mp.Isothermal)

lens = af.Model(al.Galaxy, redshift=0.5, mass=al.mp.Isothermal)

# Source:

point_0 = af.Model(al.ps.PointFlux)

source = af.Model(al.Galaxy, redshift=1.0, point_0=point_0)

# Overall Lens Model:

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))

The `info` attribute shows the model in a readable format, which now includes the `flux` parameter of the point source.

In [ ]:
print(model.info)

The same model can also be visualized as a figure, making its structure easier to understand at a glance.

The figure shows how the model is organized: which parameters belong to each component, and whether they are free, 
fixed, shared, linked by an expression, solved during the fit, or not configured. `model.info` provides the 
corresponding numerical details, including the prior assigned to each free parameter and the value of each fixed 
parameter.

In [ ]:
af.ModelPlotter(model).figure()

__Search__

The model is fitted to the data using the nested sampling algorithm Nautilus (see `start.here.py` for a 
full description).

In the `start_here.py` example 100 live points (`n_live=100`) were used to sample parameter space. We increase this
to 150, to account for the additional free parameters in the model that is the source flux.

In [ ]:
search = af.Nautilus(
    path_prefix=Path("point_source") / "features",
    name="fluxes",
    unique_tag=dataset_name,
    n_live=150,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see modeling examples.
)

__Analysis__

Create the `AnalysisPoint` object defining how the via Nautilus the model is fitted to the data.

In [ ]:
analysis = al.AnalysisPoint(
    dataset=dataset,
    solver=solver,
    image=data,  # Visualization only (`image_with_points.png`, `fit_image_with_points.png`).
    # `PointFlux` carries a free source centre, so the solved-centre default fit cannot be used
    # (it would raise an error) — we pass the free-centre all-to-all image-plane chi-squared.
    fit_positions_cls=al.FitPositionsImagePairAll,
)

__Analytic Flux (Solved)__

An alternative to sampling `flux` is to solve it analytically: `al.FitFluxesSolved` computes the
magnification-weighted best-fit source flux from the data in closed form, so the model needs no
`flux` parameter at all. The model component is then the parameter-free `al.ps.PointSolved` (which
also solves the source centre analytically — solved fits require it, and mixing solved fits with
`Point`/`PointFlux` raises an error), reducing the model above from N=8 to N=5.

The composition is shown below (not fitted here — the model-fit in this script demonstrates the
free-`flux` convention; see `guides/point_solver/pairing.py` for when to prefer each):

The microlensing caveat at the top of this script applies equally to the solved flux: solving the
source flux analytically does not make image fluxes any less affected by microlensing.

A second caveat decides between the two conventions for standardizable candles (e.g. lensed
supernovae): `FitFluxesSolved` marginalizes the source flux over a *flat* prior, so any informative
prior you hold on the intrinsic flux — the whole point of a standard candle — is discarded. For
glSNe-style science, keep the free-`flux` `PointFlux` composition demonstrated above and place your
standardization prior on `flux`.

In [ ]:
lens_solved = af.Model(al.Galaxy, redshift=0.5, mass=al.mp.Isothermal)

source_solved = af.Model(al.Galaxy, redshift=1.0, point_0=af.Model(al.ps.PointSolved))

model_solved = af.Collection(
    galaxies=af.Collection(lens=lens_solved, source=source_solved)
)

analysis_solved = al.AnalysisPoint(
    dataset=dataset,
    solver=solver,
    image=data,  # Visualization only (`image_with_points.png`, `fit_image_with_points.png`).
    fit_positions_cls=al.FitPositionsSourceSolved,  # Solved fits pair with `PointSolved`.
    fit_flux_cls=al.FitFluxesSolved,  # Flux solved analytically, no free `flux` parameter.
)

print(model_solved.info)

The solved composition is the same model with the source emptied out: `point_0 · PointSolved` has only a solved 
`centre` and no `flux` at all, because `FitFluxesSolved` computes the source flux in closed form.

In [ ]:
af.ModelPlotter(model_solved).figure()

__Run Times__

For the positions-only fit, the run time of the log likelihood function was ~0.01 seconds, which is a modest run-time.

Evaluating the time delays does not increase this much, with a value of around ~0.01 seconds still expected.

Overall modeling run times should therefore be around 20 minutes on CPU, under 5 minutes on GPU.

__Model-Fit__

We begin the model-fit by passing the model and analysis object to the non-linear search (checkout the output folder
for on-the-fly visualization and results).

In [ ]:
result = search.fit(model=model, analysis=analysis)

__Result__

The `info` attribute shows the model in a readable format (if this does not display clearly on your screen refer to
`start_here.ipynb` for a description of how to fix this).

This confirms that `flux` parameters of the source is inferred by the fit.

In [ ]:
print(result.info)

Checkout `autolens_workspace/*/guides/results` for a full description of analysing results.